# Input Parser Single Step Evaluation

Evaluates `input_parser_node` in isolation across 5 dimensions:
1. **Action Classification** ג€” correct routing decision (deterministic)
2. **Item Count** ג€” correct number of food items extracted (deterministic)
3. **Amount Accuracy** ג€” gram conversion within ֲ±20% tolerance (deterministic)
4. **Date Parsing** ג€” correct date/time extraction (deterministic)
5. **Food Name Quality** ג€” search-friendly normalization (LLM-as-judge)

In [1]:
import sys
import os
from pathlib import Path

# Add project root to path (notebooks/evals/ -> project root)
project_root = str(Path.cwd().parent.parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, ".env"))

from langsmith import Client
from langchain_core.messages import HumanMessage

from src.agents.nodes.input_node import input_parser_node

client = Client()
print("Setup complete")

2026-03-30 08:08:31 [info     ] Database backend resolved      backend='asyncpg (Supabase)'
2026-03-30 08:08:31 [info     ] LLM config loaded              model=gpt-4.1-nano provider=openai
Setup complete


## Dataset: FitPal Input Parser

~15 examples covering all 4 action types with full reference outputs.

In [2]:
examples = [
    # --- LOG_FOOD: Basic single item ---
    {
        "question": "I had 200g of chicken",
        "action": "LOG_FOOD",
        "items": [{"food_name": "Chicken", "amount": 200.0}],
        "item_count": 1,
        "consumed_at": None,
        "start_date": None,
        "end_date": None,
    },
    # --- LOG_FOOD: Multi-item ---
    {
        "question": "Log a banana and 100g rice",
        "action": "LOG_FOOD",
        "items": [
            {"food_name": "Banana", "amount": 120.0},
            {"food_name": "Rice", "amount": 100.0},
        ],
        "item_count": 2,
        "consumed_at": None,
        "start_date": None,
        "end_date": None,
    },
    # --- LOG_FOOD: No verb, just food + quantity ---
    {
        "question": "200g chicken breast",
        "action": "LOG_FOOD",
        "items": [{"food_name": "Chicken Breast", "amount": 200.0}],
        "item_count": 1,
        "consumed_at": None,
        "start_date": None,
        "end_date": None,
    },
    # --- LOG_FOOD: Single word, no quantity (default serving) ---
    {
        "question": "Coffee",
        "action": "LOG_FOOD",
        "items": [{"food_name": "Coffee", "amount": 240.0}],
        "item_count": 1,
        "consumed_at": None,
        "start_date": None,
        "end_date": None,
    },
    # --- LOG_FOOD: Meal decomposition ---
    {
        "question": "Pasta with cheese for lunch",
        "action": "LOG_FOOD",
        "items": [
            {"food_name": "Pasta", "amount": 200.0},
            {"food_name": "Cheese", "amount": 30.0},
        ],
        "item_count": 2,
        "consumed_at": None,
        "start_date": None,
        "end_date": None,
    },
    # --- LOG_FOOD: Unit conversion (cups -> grams) ---
    {
        "question": "I had 1 cup of rice",
        "action": "LOG_FOOD",
        "items": [{"food_name": "Rice", "amount": 158.0}],
        "item_count": 1,
        "consumed_at": None,
        "start_date": None,
        "end_date": None,
    },
    # --- LOG_FOOD: Unit conversion (slices -> grams) ---
    {
        "question": "2 slices of bread",
        "action": "LOG_FOOD",
        "items": [{"food_name": "Bread", "amount": 60.0}],
        "item_count": 1,
        "consumed_at": None,
        "start_date": None,
        "end_date": None,
    },
    # --- LOG_FOOD: "protein" in text should NOT confuse with stats ---
    {
        "question": "I had a protein shake after my workout",
        "action": "LOG_FOOD",
        "items": [{"food_name": "Protein Shake", "amount": 300.0}],
        "item_count": 1,
        "consumed_at": None,
        "start_date": None,
        "end_date": None,
    },
    # --- LOG_FOOD: With relative time ---
    {
        "question": "I ate 200g of chicken 2 hours ago",
        "action": "LOG_FOOD",
        "items": [{"food_name": "Chicken", "amount": 200.0}],
        "item_count": 1,
        "consumed_at": "RELATIVE",
        "start_date": None,
        "end_date": None,
    },
    # --- LOG_FOOD: With specific date ---
    {
        "question": "I had a banana yesterday",
        "action": "LOG_FOOD",
        "items": [{"food_name": "Banana", "amount": 120.0}],
        "item_count": 1,
        "consumed_at": "YESTERDAY_NOON",
        "start_date": None,
        "end_date": None,
    },
    # --- QUERY_FOOD_INFO ---
    {
        "question": "How much protein is in an egg?",
        "action": "QUERY_FOOD_INFO",
        "items": [],
        "item_count": 0,
        "consumed_at": None,
        "start_date": None,
        "end_date": None,
    },
    # --- QUERY_FOOD_INFO: Could confuse with LOG_FOOD ---
    {
        "question": "How many calories does a banana have?",
        "action": "QUERY_FOOD_INFO",
        "items": [],
        "item_count": 0,
        "consumed_at": None,
        "start_date": None,
        "end_date": None,
    },
    # --- QUERY_DAILY_STATS: Basic ---
    {
        "question": "What did I eat today?",
        "action": "QUERY_DAILY_STATS",
        "items": [],
        "item_count": 0,
        "consumed_at": None,
        "start_date": None,
        "end_date": None,
    },
    # --- QUERY_DAILY_STATS: With date range ---
    {
        "question": "Stats for last 3 days",
        "action": "QUERY_DAILY_STATS",
        "items": [],
        "item_count": 0,
        "consumed_at": None,
        "start_date": "RELATIVE_3_DAYS_AGO",
        "end_date": "TODAY",
    },
    # --- CHITCHAT ---
    {
        "question": "Hello, how are you?",
        "action": "CHITCHAT",
        "items": [],
        "item_count": 0,
        "consumed_at": None,
        "start_date": None,
        "end_date": None,
    },
]

# Dataset created in LangSmith UI under fit-pal-agent project
dataset_id = "37becfc1-a824-4b25-9204-afc090209092"
dataset_name = "Input Parser"

# Upload examples if dataset is empty
existing = list(client.list_examples(dataset_id=dataset_id))
if not existing:
    client.create_examples(
        inputs=[{"question": ex["question"]} for ex in examples],
        outputs=[{k: v for k, v in ex.items() if k != "question"} for ex in examples],
        dataset_id=dataset_id,
    )
    print(f"Uploaded {len(examples)} examples to '{dataset_name}'")
else:
    print(f"Dataset '{dataset_name}' already has {len(existing)} examples")

Dataset 'Input Parser' already has 15 examples


## Target Function

Calls `input_parser_node` directly with a minimal state dict.

In [3]:
async def run_input_parser(inputs: dict) -> dict:
    """Run input_parser_node and return structured outputs for evaluation."""
    state = {"messages": [HumanMessage(content=inputs["question"])]}
    result = input_parser_node(state)
    return {
        "action": result["last_action"],
        "items": result["pending_food_items"],
        "item_count": len(result["pending_food_items"]),
        "consumed_at": str(result["consumed_at"]) if result.get("consumed_at") else None,
        "start_date": str(result["start_date"]) if result.get("start_date") else None,
        "end_date": str(result["end_date"]) if result.get("end_date") else None,
    }

In [4]:
# Smoke test ג€” verify target function works before running full eval
test_result = await run_input_parser({"question": "I had 200g of chicken"})
print(test_result)

2026-03-30 08:08:38 [info     ] Input parsed                   action=LOG_FOOD items=1
{'action': 'LOG_FOOD', 'items': [{'food_name': 'Chicken', 'amount': 200.0, 'unit': 'g', 'original_text': '200g of chicken'}], 'item_count': 1, 'consumed_at': None, 'start_date': None, 'end_date': None}


## Evaluators

5 evaluators, each checking one dimension of the parser output.

In [5]:
def correct_action(outputs: dict, reference_outputs: dict) -> bool:
    """Check if the parser selected the correct action/route."""
    return outputs["action"] == reference_outputs["action"]

In [6]:
def correct_item_count(outputs: dict, reference_outputs: dict) -> bool:
    """Check if the parser extracted the correct number of food items."""
    return outputs["item_count"] == reference_outputs["item_count"]

In [7]:
def amount_accuracy(outputs: dict, reference_outputs: dict) -> dict:
    """Check if extracted amounts are within +/-20% of expected values.

    Returns a score between 0.0 and 1.0 (fraction of items within tolerance).
    Skips if no items expected (non-food actions).
    """
    expected_items = reference_outputs.get("items", [])
    actual_items = outputs.get("items", [])

    if not expected_items:
        return {"key": "amount_accuracy", "score": 1.0, "comment": "No items to check"}

    if len(actual_items) != len(expected_items):
        return {
            "key": "amount_accuracy",
            "score": 0.0,
            "comment": f"Item count mismatch: got {len(actual_items)}, expected {len(expected_items)}",
        }

    # Sort both lists by food_name for alignment
    expected_sorted = sorted(expected_items, key=lambda x: x["food_name"].lower())
    actual_sorted = sorted(actual_items, key=lambda x: x["food_name"].lower())

    within_tolerance = 0
    details = []
    for exp, act in zip(expected_sorted, actual_sorted):
        exp_amount = exp["amount"]
        act_amount = act["amount"]
        tolerance = exp_amount * 0.20
        is_close = abs(act_amount - exp_amount) <= tolerance
        if is_close:
            within_tolerance += 1
        details.append(
            f"{act.get('food_name', '?')}: {act_amount}g vs {exp_amount}g"
            f" ({'OK' if is_close else 'FAIL'})"
        )

    score = within_tolerance / len(expected_items)
    return {"key": "amount_accuracy", "score": score, "comment": "; ".join(details)}

In [8]:
from datetime import date, datetime, timedelta


def _resolve_date_sentinel(sentinel: str | None) -> str | None:
    """Convert sentinel values to actual date strings at eval time."""
    if sentinel is None:
        return None
    today = date.today()
    mapping = {
        "TODAY": str(today),
        "YESTERDAY_NOON": str(
            datetime.combine(
                today - timedelta(days=1),
                datetime.min.replace(hour=12).time(),
            )
        ),
        # "last 3 days" inclusive of today = today, yesterday, day before = today - 2
        "RELATIVE_3_DAYS_AGO": str(today - timedelta(days=2)),
    }
    if sentinel in mapping:
        return mapping[sentinel]
    if sentinel == "RELATIVE":
        return "RELATIVE"  # Special case: just check it's not None
    return sentinel


def _dates_equivalent(expected: str | None, actual: str | None) -> bool:
    """Compare date values, treating null and today as equivalent.

    For QUERY_DAILY_STATS without explicit date, both null and today are valid.
    The model may return start_date=today, end_date=today OR leave them null.
    Both are correct — stats_node handles both paths identically.
    """
    if expected is None and actual is None:
        return True
    if expected is not None and actual is not None:
        return expected[:10] == actual[:10]
    # One is None, one is not — accept if the non-None value is today
    today_str = str(date.today())
    if expected is None and actual is not None:
        return actual[:10] == today_str
    if expected is not None and actual is None:
        return expected[:10] == today_str
    return False


def correct_dates(outputs: dict, reference_outputs: dict) -> bool:
    """Check if date/time extraction matches expected values.

    Handles sentinel values for relative dates.
    For 'RELATIVE' consumed_at: just checks it's not None.
    For specific dates: checks date portion matches.
    Treats null and today as equivalent for start_date/end_date.
    """
    # --- consumed_at ---
    expected_consumed = _resolve_date_sentinel(reference_outputs.get("consumed_at"))
    actual_consumed = outputs.get("consumed_at")

    if expected_consumed == "RELATIVE":
        if actual_consumed is None:
            return False
    elif expected_consumed is not None:
        if actual_consumed is None:
            return False
        if expected_consumed[:10] != actual_consumed[:10]:
            return False
    else:
        if actual_consumed is not None:
            return False

    # --- start_date (null and today are equivalent) ---
    expected_start = _resolve_date_sentinel(reference_outputs.get("start_date"))
    actual_start = outputs.get("start_date")
    if not _dates_equivalent(expected_start, actual_start):
        return False

    # --- end_date (null and today are equivalent) ---
    expected_end = _resolve_date_sentinel(reference_outputs.get("end_date"))
    actual_end = outputs.get("end_date")
    if not _dates_equivalent(expected_end, actual_end):
        return False

    return True

In [9]:
from typing import Annotated

from langchain.chat_models import init_chat_model
from typing_extensions import TypedDict


class NameGrade(TypedDict):
    """Grade for food name normalization quality."""

    reasoning: Annotated[
        str, ..., "Step-by-step reasoning for the grade."
    ]
    is_acceptable: Annotated[
        bool, ..., "True if the name is a reasonable search-friendly normalization."
    ]


judge_instructions = """You are evaluating whether a food name has been properly normalized for database search.

Rules:
- The name should be generic and search-friendly (e.g., "Apple" not "Small sour green apple")
- Minor variations are acceptable ("Chicken" vs "Chicken Breast" - both valid)
- The name must still refer to the same food as the original text
- Compound dishes should be decomposed ("Pasta with cheese" -> "Pasta" and "Cheese" separately)
- Individual ingredients from decomposed dishes are valid on their own ("Cheese" from "pasta with cheese" is acceptable)
- Common food product names are acceptable even if multi-word ("Protein Shake", "Peanut Butter", "Greek Yogurt")
- Do NOT penalize names that are already standard food category names

Grade as acceptable if a nutrition database search for this name would reasonably find the correct food."""

judge_llm = init_chat_model("gpt-4o", temperature=0).with_structured_output(
    NameGrade, method="json_schema"
)


async def food_name_quality(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """LLM-as-judge: evaluate if food names are reasonable normalizations.

    Skips if no items expected (non-food actions). Returns score 0.0-1.0
    (fraction of names graded acceptable).
    """
    expected_items = reference_outputs.get("items", [])
    actual_items = outputs.get("items", [])

    if not expected_items:
        return {"key": "food_name_quality", "score": 1.0, "comment": "No items to check"}

    if not actual_items:
        return {"key": "food_name_quality", "score": 0.0, "comment": "No items produced"}

    acceptable_count = 0
    details = []

    for act in actual_items:
        msg = (
            f'Original user input: "{inputs["question"]}"\n'
            f'Extracted food name: "{act.get("food_name", "")}"\n'
            f'\nIs this a reasonable, search-friendly normalization?'
        )

        grade = await judge_llm.ainvoke([
            {"role": "system", "content": judge_instructions},
            {"role": "user", "content": msg},
        ])

        if grade["is_acceptable"]:
            acceptable_count += 1
        details.append(
            f"{act.get('food_name', '?')}: "
            f"{'OK' if grade['is_acceptable'] else 'FAIL'} - "
            f"{grade['reasoning'][:80]}"
        )

    score = acceptable_count / len(actual_items)
    return {"key": "food_name_quality", "score": score, "comment": "; ".join(details)}

## Run Evaluation

Execute all evaluators against the dataset and display results.

In [10]:
experiment_results = await client.aevaluate(
    run_input_parser,
    data=dataset_id,
    evaluators=[
        correct_action,
        correct_item_count,
        amount_accuracy,
        correct_dates,
        food_name_quality,
    ],
    experiment_prefix="input-parser-eval",
    max_concurrency=4,
)
experiment_results.to_pandas()

c:\Users\User\Desktop\fit_pal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'input-parser-eval-928200ce' at:
https://smith.langchain.com/o/df0d054f-18d0-4ce1-9c1e-f8bdd89430f4/datasets/37becfc1-a824-4b25-9204-afc090209092/compare?selectedSessions=8372300a-5368-4aef-9355-8282c9dc5b2c




0it [00:00, ?it/s]

2026-03-30 08:08:44 [info     ] Input parsed                   action=LOG_FOOD items=1
2026-03-30 08:08:45 [info     ] Input parsed                   action=QUERY_DAILY_STATS items=0
2026-03-30 08:08:46 [info     ] Input parsed                   action=CHITCHAT items=0
2026-03-30 08:08:47 [info     ] Input parsed                   action=QUERY_FOOD_INFO items=0


1it [00:05,  5.26s/it]

...

2026-03-30 08:09:11 [info     ] Input parsed                   action=LOG_FOOD items=1


11it [00:30,  2.12s/it]

2026-03-30 08:09:13 [info     ] Input parsed                   action=QUERY_FOOD_INFO items=0


15it [00:32,  2.16s/it]


,inputs.question,outputs.action,outputs.items,outputs.item_count,outputs.consumed_at,outputs.start_date,outputs.end_date,error,reference.items,reference.action,...,reference.start_date,reference.consumed_at,feedback.correct_action,feedback.correct_item_count,feedback.amount_accuracy,feedback.correct_dates,feedback.food_name_quality,execution_time,example_id,id
0,Stats for last 3 days,QUERY_DAILY_STATS,[],0,NaN,2026-03-27,2026-03-29,None,[],QUERY_DAILY_STATS,...,RELATIVE_3_DAYS_AGO,NaN,True,True,1.0,False,1.0,1.073094,344b906b-75fb-4433-a8ef-c4b624454301,019d3d24-efe1-7180-b461-64d4504450fc
1,"Hello, how are you?",CHITCHAT,[],0,NaN,NaN,NaN,None,[],CHITCHAT,...,NaN,NaN,True,True,1.0,True,1.0,2.082793,4aa99ce2-433d-4f66-974c-b8bd77f99126,019d3d24-efe3-7e32-a84f-c7febbe44006
2,How many calories does a banana have?,QUERY_FOOD_INFO,[],0,NaN,NaN,NaN,None,[],QUERY_FOOD_INFO,...,NaN,NaN,True,True,1.0,True,1.0,0.950417,5906ce00-40f4-458c-a265-1a958413e5f5,019d3d24-f806-7460-a707-91156916ace4
3,I had 1 cup of rice,LOG_FOOD,"[{'food_name': 'Rice', 'amount': 158.0, 'unit'...",1,NaN,NaN,NaN,None,"[{'amount': 158.0, 'food_name': 'Rice'}]",LOG_FOOD,...,NaN,NaN,True,True,1.0,True,1.0,3.042273,7301e1f3-5bd9-4b36-aa05-e7257c3eab6a,019d3d25-0029-79a0-beed-0834554ec566
4,What did I eat today?,QUERY_DAILY_STATS,[],0,NaN,NaN,NaN,None,[],QUERY_DAILY_STATS,...,NaN,NaN,True,True,1.0,True,1.0,4.924050,7d0bd410-07f9-4a6c-8a44-39555b399163,019d3d25-2108-7332-887f-6898868f098a
5,I had a banana yesterday,LOG_FOOD,"[{'food_name': 'Banana', 'amount': 118.0, 'uni...",1,2026-03-29 12:00:00+00:00,NaN,NaN,None,"[{'amount': 120.0, 'food_name': 'Banana'}]",LOG_FOOD,...,NaN,YESTERDAY_NOON,True,True,1.0,True,1.0,2.184490,1275e9da-2e04-400a-841e-04283f50e55e,019d3d24-e759-7801-a885-8a2d4cd55230
6,I had a protein shake after my workout,LOG_FOOD,"[{'food_name': 'Protein Shake', 'amount': 250....",1,NaN,NaN,NaN,None,"[{'amount': 300.0, 'food_name': 'Protein Shake'}]",LOG_FOOD,...,NaN,NaN,True,True,1.0,True,1.0,1.089433,604ea078-6829-4d6b-aef8-2cded883bebc,019d3d24-fbe2-7df0-8310-eab55465387b
7,2 slices of bread,LOG_FOOD,"[{'food_name': 'Bread', 'amount': 60.0, 'unit'...",1,NaN,NaN,NaN,None,"[{'amount': 60.0, 'food_name': 'Bread'}]",LOG_FOOD,...,NaN,NaN,True,True,1.0,True,1.0,1.228383,63e29232-caa0-4ca1-8467-71595de7b061,019d3d25-0027-7c21-b1cb-bf2ead22bf01
8,I ate 200g of chicken 2 hours ago,LOG_FOOD,"[{'food_name': 'Chicken', 'amount': 200.0, 'un...",1,2026-03-30 06:09:06+00:00,NaN,NaN,None,"[{'amount': 200.0, 'food_name': 'Chicken'}]",LOG_FOOD,...,NaN,RELATIVE,True,True,1.0,True,1.0,3.014075,d3c03e7c-a06e-4860-b637-a102d46cd198,019d3d25-4039-76c3-8d3e-75bb60bac892
9,200g chicken breast,LOG_FOOD,"[{'food_name': 'Chicken Breast', 'amount': 200...",1,NaN,NaN,NaN,None,"[{'amount': 200.0, 'food_name': 'Chicken Breas...",LOG_FOOD,...,NaN,NaN,True,True,1.0,True,1.0,1.164707,b0a4c458-04af-446b-a515-ed3718259c48,019d3d25-3bad-76d0-98a3-762a8a3d6c88


## Notes

- Re-run this notebook after changing the input parser prompt or switching LLM models
- Compare experiments in LangSmith UI: Datasets & Testing -> FitPal: Input Parser
- To add more examples, update the `examples` list and delete/recreate the dataset
- Amount tolerance is +/-20% ג€” adjust in `amount_accuracy` if too strict/lenient